# Melodix — Stage 2 training (Colab)

**This notebook calls `scripts/train_yolo.py`. It does not call `model.train()`.**

That distinction is the entire point of the notebook. The first checkpoint was
trained by an ad-hoc `model.train()` cell, which silently ran on ultralytics'
stock augmentation: `fliplr=0.5` mirrored half the training pages, `mosaic=1.0`
quartered the effective resolution, and `degrees=0.0` gave no exposure to the
residual tilt deskew leaves behind. None of that was visible in the metrics, and
it was only found by reading `train_args` out of the checkpoint months later.

Configuration that lives in a script nobody calls is documentation, not
configuration. Every cell below routes through the script so the settings that
actually run are the settings that are reviewed.


## 1. Check the GPU

Runtime → Change runtime type → T4 GPU. A 30-epoch run at `imgsz=1280` takes
roughly 100 minutes on a T4 and roughly 60 hours on CPU, so this matters.


In [ ]:
!nvidia-smi


## 2. Get the repository

Replace the URL with your remote, or upload a zip and unpack it instead.


In [ ]:
REPO = 'https://github.com/melodix/melodix.git'  # or your fork
!git clone $REPO /content/melodix || echo 'already cloned'
%cd /content/melodix
!git log --oneline -1


## 3. Install

`fix_opencv.py` is required: ultralytics pulls the GUI OpenCV build, this
project needs the headless one, and pip installs both over the same files.


In [ ]:
!pip install -q -e '.[vision]'
!python scripts/fix_opencv.py
!python scripts/fix_opencv.py --check


## 4. Get the dataset

Either generate it here (reproducible from the seed) or mount Drive and copy a
snapshot. Generating is preferred: the snapshot the first checkpoint trained on
was never preserved, which is why its metrics cannot be reproduced exactly.


In [ ]:
# Option A — generate (reproducible from the seed)
!python scripts/generate_synthetic_dataset.py \
    --out /content/datasets/melodix_synth --pages 2500 --seed 0

# Option B — from Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/melodix_datasets/melodix_synth /content/datasets/


## 5. Verify the labels before spending an hour on them

Checks that boxes sit on glyphs. The radius correlation is the one that matters:
a label set left behind by a rotation is displaced in proportion to distance
from the page centre.


In [ ]:
!python scripts/verify_labels.py --data /content/datasets/melodix_synth --sample 80


## 6. Review the resolved config, then train

`--check-only` prints every setting that will be used and trains nothing. Read
it before starting the run — this is the review step that was missing the first
time. Note `fliplr = 0.0` and `mosaic = 0.0`.


In [ ]:
!python scripts/train_yolo.py \
    --data /content/datasets/melodix_synth/data.yaml \
    --output-dir /content/drive/MyDrive/melodix_runs \
    --name stage2_corrected_aug \
    --check-only


### Train

`--name` is deliberately not `stage2_synth`: **the existing checkpoint is the
baseline and must not be overwritten**, or the comparison this run exists to
make becomes impossible.


In [ ]:
!python scripts/train_yolo.py \
    --data /content/datasets/melodix_synth/data.yaml \
    --output-dir /content/drive/MyDrive/melodix_runs \
    --name stage2_corrected_aug \
    --epochs 30 --batch 8 --workers 8 --device 0


## 7. Confirm what actually ran

Two records, and they should agree. `melodix_resolved_config.json` is written by
the script before training; `train_args` is written by ultralytics into the
checkpoint. A run that bypassed the script has the second and not the first.


In [ ]:
import json, torch
run = '/content/drive/MyDrive/melodix_runs/stage2_corrected_aug'

print('--- melodix_resolved_config.json ---')
cfg = json.load(open(f'{run}/melodix_resolved_config.json'))
for k in ('fliplr','flipud','mosaic','degrees','imgsz','max_det'):
    print(f'  {k:10s} {cfg["settings"][k]}')

print('--- train_args inside best.pt ---')
ck = torch.load(f'{run}/weights/best.pt', map_location='cpu', weights_only=False)
for k in ('fliplr','flipud','mosaic','degrees','imgsz','max_det','epochs'):
    print(f'  {k:10s} {ck["train_args"].get(k)}')
assert ck['train_args']['fliplr'] == 0.0, 'mirror augmentation was applied'
print('OK: no mirror augmentation')


## 8. Validate, and keep the baseline

Download both checkpoints. Record the new run in `models/PROVENANCE.md` **beside**
the baseline, not replacing it.


In [ ]:
!python scripts/validate_yolo.py \
    --weights /content/drive/MyDrive/melodix_runs/stage2_corrected_aug/weights/best.pt \
    --data /content/datasets/melodix_synth/data.yaml \
    --imgsz 1280 --json /content/validation_corrected_aug.json
